# Mundo do Aspirador — Ambiente, Simulação e Busca

Modelagem completa do ambiente (estado, ações, custo) e resolução do mesmo problema por busca cega e busca de custo uniforme, com comparação entre as estratégias.

**Técnica:** BFS, DFS e busca de custo uniforme  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/02-mundo-do-aspirador-busca.ipynb)


In [2]:
# ============================================================
# MUNDO DO ASPIRADOR DE PÓ
# Células 1, 2 e 3: Ambiente, Simulação e Busca
# ============================================================

from collections import deque
import heapq
import math

# ============================================================
# CÉLULA 1 — Classe do Ambiente (4 componentes do problema)
# ============================================================

class Estado:
    """
    Representa um estado do mundo do aspirador.
    Componente 1: Estado = (posição, sujeira_esq, sujeira_dir)
    """
    def __init__(self, posicao, sujeira_esq, sujeira_dir):
        self.posicao     = posicao       # 'E' ou 'D'
        self.sujeira_esq = sujeira_esq   # True = sujo, False = limpo
        self.sujeira_dir = sujeira_dir

    def __eq__(self, other):
        return (self.posicao     == other.posicao and
                self.sujeira_esq == other.sujeira_esq and
                self.sujeira_dir == other.sujeira_dir)

    def __hash__(self):
        return hash((self.posicao, self.sujeira_esq, self.sujeira_dir))

    def __repr__(self):
        e = "S" if self.sujeira_esq else "L"
        d = "S" if self.sujeira_dir else "L"
        return f"(pos={self.posicao}, E={e}, D={d})"

    def copia(self):
        return Estado(self.posicao, self.sujeira_esq, self.sujeira_dir)


class AmbienteAspirador:
    """
    Implementa os 4 componentes do problema:
      1. Estado inicial  → definido no construtor
      2. Ações           → metodo acoes_possiveis()
      3. Teste objetivo  → metodo eh_objetivo()
      4. Custo de caminho→ cada ação custa 1 (fixo)
    """

    ACOES = ['Aspirar', 'Mover_Esquerda', 'Mover_Direita']
    CUSTO_PASSO = 1  # Componente 4: custo uniforme por ação

    def __init__(self, posicao='E', sujeira_esq=True, sujeira_dir=True):
        # Componente 1: Estado Inicial
        self.estado_inicial = Estado(posicao, sujeira_esq, sujeira_dir)

    def acoes_possiveis(self, estado):
        """
        Componente 2: Retorna ações aplicáveis ao estado.
        Mover_Esquerda só faz sentido se não está na esquerda (e vice-versa),
        mas permitimos todas para mostrar ações sem efeito.
        """
        return self.ACOES  # Todas sempre disponíveis

    def transicao(self, estado, acao):
        """
        Modelo de transição: retorna NOVO estado após aplicar ação.
        O estado original não é modificado (imutabilidade).
        """
        novo = estado.copia()

        if acao == 'Aspirar':
            # Aspira a célula atual (mesmo que já limpa — sem efeito)
            if novo.posicao == 'E':
                novo.sujeira_esq = False   # ← Q3b: já limpa? continua False
            else:
                novo.sujeira_dir = False

        elif acao == 'Mover_Esquerda':
            novo.posicao = 'E'            # Vai para esquerda (mesmo que já lá)

        elif acao == 'Mover_Direita':
            novo.posicao = 'D'            # Vai para direita  (mesmo que já lá)

        return novo

    def eh_objetivo(self, estado):
        """
        Componente 3: Teste de objetivo.
        Objetivo atingido quando AMBAS as células estão limpas.
        """
        return not estado.sujeira_esq and not estado.sujeira_dir

    def custo_caminho(self, acoes):
        """Componente 4: Custo total = número de ações × 1."""
        return len(acoes) * self.CUSTO_PASSO

    def todos_os_estados(self):
        """Enumera todos os 8 estados possíveis do mundo."""
        estados = []
        for pos in ['E', 'D']:
            for se in [True, False]:
                for sd in [True, False]:
                    estados.append(Estado(pos, se, sd))
        return estados


def sep(char='=', n=60):
    print(char * n)


def mostrar_estado(estado, ambiente, passo=None, acao=None, custo=None):
    """Exibe estado de forma visual."""
    e_icon = "💩" if estado.sujeira_esq else "✨"
    d_icon = "💩" if estado.sujeira_dir else "✨"
    rob_e  = "🤖" if estado.posicao == 'E' else "  "
    rob_d  = "🤖" if estado.posicao == 'D' else "  "
    obj    = " ← 🏆 OBJETIVO!" if ambiente.eh_objetivo(estado) else ""

    prefixo = f"Passo {passo:2d} | {acao:<17} | Custo {custo} |" if passo else "Estado Inicial        |"
    print(f"  {prefixo}  [{rob_e}{e_icon} Esq] [{d_icon}{rob_d} Dir]{obj}")


# ============================================================
# CÉLULA 2 — Simulação com Agente Reativo
# ============================================================

def agente_reativo(estado, ambiente):
    """
    Estratégia simples: aspira se sujo, move se limpo.
    Q4b: Esta estratégia é ótima? Analisamos a seguir.
    """
    if estado.posicao == 'E' and estado.sujeira_esq:
        return 'Aspirar'
    elif estado.posicao == 'D' and estado.sujeira_dir:
        return 'Aspirar'
    elif estado.posicao == 'E':
        return 'Mover_Direita'
    else:
        return 'Mover_Esquerda'


def simular(posicao='E', sujeira_esq=True, sujeira_dir=True, max_passos=10):
    """
    Célula 2: Simulação completa a partir de um estado inicial.
    Responde Q4a: passo a passo com análise de otimalidade.
    """
    ambiente = AmbienteAspirador(posicao, sujeira_esq, sujeira_dir)
    estado   = ambiente.estado_inicial.copia()

    sep()
    print(f"SIMULAÇÃO — Estado inicial: {estado}")
    sep()
    mostrar_estado(estado, ambiente)

    for passo in range(1, max_passos + 1):
        if ambiente.eh_objetivo(estado):
            break
        acao   = agente_reativo(estado, ambiente)
        estado = ambiente.transicao(estado, acao)
        custo  = passo
        mostrar_estado(estado, ambiente, passo, acao, custo)

    sep('-', 60)
    status = "✅ Objetivo alcançado" if ambiente.eh_objetivo(estado) else "❌ Falhou"
    print(f"  Resultado: {status} em {passo if ambiente.eh_objetivo(estado) else '?'} passo(s)")


def analisar_estrategia():
    """
    Q4b: Testa a estratégia em todos os 8 estados iniciais.
    Mostra onde ela é ótima e onde falha.
    """
    sep()
    print("ANÁLISE DA ESTRATÉGIA EM TODOS OS ESTADOS INICIAIS")
    sep()
    print(f"  {'Estado Inicial':<30} {'Passos':>6}  {'Ótimo?':>8}  Caminho")
    print("  " + "-" * 70)

    ambiente_ref = AmbienteAspirador()
    for estado_ini in ambiente_ref.todos_os_estados():
        ambiente = AmbienteAspirador(estado_ini.posicao,
                                     estado_ini.sujeira_esq,
                                     estado_ini.sujeira_dir)
        estado = estado_ini.copia()
        caminho = []

        for _ in range(20):
            if ambiente.eh_objetivo(estado):
                break
            acao   = agente_reativo(estado, ambiente)
            estado = ambiente.transicao(estado, acao)
            caminho.append(acao[0])  # Inicial da ação para brevidade

        passos = len(caminho)
        # Ótimo: 0 passos se já objetivo, 1 se 1 célula suja, 2 se as 2 sujas
        sujas = estado_ini.sujeira_esq + estado_ini.sujeira_dir
        otimo_esperado = sujas if estado_ini.posicao in (
            ['E'] if estado_ini.sujeira_esq else ['D']
        ) else sujas + (1 if sujas > 0 else 0)

        otimo = "✅" if passos <= otimo_esperado else "⚠️ +1"
        cam_str = "→".join(caminho) if caminho else "(já limpo)"
        print(f"  {str(estado_ini):<30} {passos:>6}  {otimo:>8}  {cam_str}")


# ============================================================
# CÉLULA 3 — Busca no Espaço de Estados
# ============================================================

class No:
    """Nó da árvore de busca."""
    def __init__(self, estado, pai=None, acao=None, custo=0):
        self.estado = estado
        self.pai    = pai
        self.acao   = acao
        self.custo  = custo

    def caminho(self):
        """Reconstrói sequência de ações até este nó."""
        nos, acoes = [], []
        no = self
        while no.pai:
            acoes.append(no.acao)
            no = no.pai
        return list(reversed(acoes))

    def __lt__(self, other):   # Para heapq no UCS
        return self.custo < other.custo


def bfs_aspirador(ambiente):
    """
    BFS: Busca em Largura — garante menor número de passos.
    Q5a/b: Expande nós por nível (camada por camada).
    """
    no_inicial = No(ambiente.estado_inicial)
    if ambiente.eh_objetivo(no_inicial.estado):
        return no_inicial, 0

    fronteira     = deque([no_inicial])
    explorados    = set()
    nos_gerados   = 0

    while fronteira:
        no = fronteira.popleft()
        explorados.add(no.estado)

        for acao in ambiente.acoes_possiveis(no.estado):
            filho_estado = ambiente.transicao(no.estado, acao)
            nos_gerados += 1
            filho = No(filho_estado, no, acao, no.custo + 1)

            if filho_estado not in explorados and filho_estado not in [f.estado for f in fronteira]:
                if ambiente.eh_objetivo(filho_estado):
                    return filho, nos_gerados
                fronteira.append(filho)

    return None, nos_gerados


def dfs_aspirador(ambiente, limite=50):
    """
    DFS: Busca em Profundidade.
    Q5c: NÃO garante solução ótima — depende da ordem de expansão.
    """
    no_inicial  = No(ambiente.estado_inicial)
    fronteira   = [no_inicial]
    explorados  = set()
    nos_gerados = 0

    while fronteira:
        no = fronteira.pop()
        if ambiente.eh_objetivo(no.estado):
            return no, nos_gerados
        if no.estado in explorados or no.custo > limite:
            continue
        explorados.add(no.estado)

        for acao in reversed(ambiente.acoes_possiveis(no.estado)):
            filho_estado = ambiente.transicao(no.estado, acao)
            nos_gerados += 1
            if filho_estado not in explorados:
                fronteira.append(No(filho_estado, no, acao, no.custo + 1))

    return None, nos_gerados


def ucs_aspirador(ambiente):
    """
    UCS: Busca de Custo Uniforme.
    Q5a: Com custo=1 por passo, equivale ao BFS.
    """
    no_inicial  = No(ambiente.estado_inicial)
    fronteira   = [(0, no_inicial)]
    explorados  = {}
    nos_gerados = 0
    contador    = 0

    while fronteira:
        custo, no = heapq.heappop(fronteira)
        if ambiente.eh_objetivo(no.estado):
            return no, nos_gerados
        if no.estado in explorados and explorados[no.estado] <= custo:
            continue
        explorados[no.estado] = custo

        for acao in ambiente.acoes_possiveis(no.estado):
            filho_estado = ambiente.transicao(no.estado, acao)
            novo_custo   = custo + ambiente.CUSTO_PASSO
            nos_gerados += 1
            if filho_estado not in explorados or explorados[filho_estado] > novo_custo:
                contador += 1
                heapq.heappush(fronteira, (novo_custo,
                               No(filho_estado, no, acao, novo_custo)))

    return None, nos_gerados


def busca_todos_estados(algoritmo_fn, nome):
    """
    Executa busca a partir de todos os 8 estados iniciais.
    Responde Q5b: quantos nós são gerados no pior caso.
    """
    sep()
    print(f"BUSCA — {nome} — Todos os Estados Iniciais")
    sep()
    print(f"  {'Estado Inicial':<28} {'Passos':>6}  {'Nós':>6}  Solução")
    print("  " + "-" * 65)

    ambiente_ref = AmbienteAspirador()
    pior_nos = 0

    for est in ambiente_ref.todos_os_estados():
        amb = AmbienteAspirador(est.posicao, est.sujeira_esq, est.sujeira_dir)
        resultado, nos = algoritmo_fn(amb)

        if resultado:
            acoes  = resultado.caminho()
            passos = len(acoes)
            sol    = "→".join(a[:3] for a in acoes) if acoes else "(já objetivo)"
        else:
            passos, sol = -1, "Sem solução"

        if nos > pior_nos:
            pior_nos = nos

        print(f"  {str(est):<28} {passos:>6}  {nos:>6}  {sol}")

    print(f"\n  ⚠️  Pior caso — nós gerados: {pior_nos}")


# ============================================================
# CÉLULA 4 — Generalização (Questão 6)
# ============================================================

def generalizar_mundo():
    sep()
    print("GENERALIZAÇÃO: N CÉLULAS")
    sep()
    print(f"\n  {'N células':>10}  {'Posições':>10}  {'Configs sujeira':>16}  {'Total estados':>14}")
    print("  " + "-" * 56)

    for n in [1, 2, 3, 4, 5, 8, 10, 15, 20]:
        posicoes  = n
        configs   = 2 ** n
        total     = posicoes * configs
        t_str     = f"{total:,}" if total < 1_000_000 else f"{total:.2e}"
        print(f"  {n:>10}  {posicoes:>10}  {configs:>16,}  {t_str:>14}")

    print("""
  📌 Fórmula geral:  N × 2^N  estados

  🧠 Explosão Combinatória:
     • N=2  →        8 estados  (resolve-se trivialmente)
     • N=10 →   10.240 estados  (ainda tratável)
     • N=20 →  ~20 milhões      (começa a ser custoso)
     • N=30 →  ~32 bilhões      (impraticável por força bruta)

  💡 Isso motiva o uso de heurísticas e busca informada
     (A*, greedy) em vez de busca cega para mundos grandes!
""")


# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

def main():
    sep()
    print("  MUNDO DO ASPIRADOR DE PÓ — Solução Completa")
    print("  Células 1, 2 e 3: Ambiente · Simulação · Busca")
    sep()

    # ── Célula 1: Listar todos os estados ───────────────────
    sep()
    print("CÉLULA 1 — Os 8 Estados Possíveis")
    sep()
    amb = AmbienteAspirador()
    print(f"  {'#':>3}  {'Estado':<30}  {'Objetivo?'}")
    print("  " + "-" * 48)
    for i, est in enumerate(amb.todos_os_estados(), 1):
        obj = "✅ SIM" if amb.eh_objetivo(est) else "❌ não"
        print(f"  {i:>3}  {str(est):<30}  {obj}")

    # ── Célula 2: Simulação ──────────────────────────────────
    print()
    simular('E', True, True)   # Q4a: pior caso
    print()
    simular('D', False, False)  # Estado já objetivo
    print()
    analisar_estrategia()

    # ── Célula 3: Busca ──────────────────────────────────────
    print()
    busca_todos_estados(bfs_aspirador, "BFS  (Largura)")
    print()
    busca_todos_estados(ucs_aspirador, "UCS  (Custo Uniforme)")
    print()
    busca_todos_estados(dfs_aspirador, "DFS  (Profundidade)")

    # ── Célula 3 Extra: Comparação BFS vs UCS ───────────────
    sep()
    print("Q5a — COMPARAÇÃO BFS × UCS (estado mais difícil)")
    sep()
    for estado_ini in [Estado('D', True, True), Estado('E', False, True)]:
        print(f"\n  Estado: {estado_ini}")
        for nome, fn in [("BFS", bfs_aspirador), ("UCS", ucs_aspirador), ("DFS", dfs_aspirador)]:
            a = AmbienteAspirador(estado_ini.posicao, estado_ini.sujeira_esq, estado_ini.sujeira_dir)
            res, nos = fn(a)
            passos = len(res.caminho()) if res else "?"
            acoes  = "→".join(res.caminho()) if res else "Sem solução"
            print(f"    {nome}: {passos} passo(s), {nos} nós  →  {acoes}")

    # ── Célula 4: Generalização ──────────────────────────────
    generalizar_mundo()

    sep()
    print("FIM DA EXECUÇÃO")
    sep()


if __name__ == "__main__":
    main()

  MUNDO DO ASPIRADOR DE PÓ — Solução Completa
  Células 1, 2 e 3: Ambiente · Simulação · Busca
CÉLULA 1 — Os 8 Estados Possíveis
    #  Estado                          Objetivo?
  ------------------------------------------------
    1  (pos=E, E=S, D=S)               ❌ não
    2  (pos=E, E=S, D=L)               ❌ não
    3  (pos=E, E=L, D=S)               ❌ não
    4  (pos=E, E=L, D=L)               ✅ SIM
    5  (pos=D, E=S, D=S)               ❌ não
    6  (pos=D, E=S, D=L)               ❌ não
    7  (pos=D, E=L, D=S)               ❌ não
    8  (pos=D, E=L, D=L)               ✅ SIM

SIMULAÇÃO — Estado inicial: (pos=E, E=S, D=S)
  Estado Inicial        |  [🤖💩 Esq] [💩   Dir]
  Passo  1 | Aspirar           | Custo 1 |  [🤖✨ Esq] [💩   Dir]
  Passo  2 | Mover_Direita     | Custo 2 |  [  ✨ Esq] [💩🤖 Dir]
  Passo  3 | Aspirar           | Custo 3 |  [  ✨ Esq] [✨🤖 Dir] ← 🏆 OBJETIVO!
------------------------------------------------------------
  Resultado: ✅ Objetivo alcançado em 4 passo(s)

SIMUL